# Problem 98: Workflow Checkpointing

**Difficulty:** Hard | **Framework:** OpenAI Agents SDK

**Categories:** orchestration, error-recovery

This notebook accompanies [Agent Foundry](https://agent-foundry-pi.vercel.app/problems/workflow-checkpointing).

## 1. Install Dependencies

In [ ]:
!pip install -q openai-agents

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Constraints

- The workflow must checkpoint state after each successful step
- On failure, the workflow must resume from the last successful checkpoint, not restart from step 1
- Checkpoints must persist across process restarts (not just in-memory)
- The resumed workflow must produce the same final result as an uninterrupted run


## 4. Starter Code (has a bug — fix it!)

The code below has an intentional issue. Read the problem description and fix it.

In [ ]:
from agents import Agent, Runner

# BUG: No checkpointing — if step 4 fails, the entire 5-step pipeline restarts from scratch
# TODO: Add checkpointing so the workflow resumes from the failed step
def run_pipeline(data: str) -> str:
    steps = [
        "Extract key entities from the data",
        "Classify the document type",
        "Summarize the main findings",
        "Generate recommendations based on findings",
        "Format the final report",
    ]
    results = []

    for i, step_instruction in enumerate(steps):
        print(f"Running step {i+1}: {step_instruction}")

        # Simulate occasional failure at step 4
        if i == 3 and not hasattr(run_pipeline, '_retried'):
            run_pipeline._retried = True
            raise RuntimeError(f"Step {i+1} failed: API timeout")

        context = "\n".join(results) if results else data
        agent = Agent(
            name=f"Step {i+1} Processor",
            instructions=f"Step {i+1}: {step_instruction}",
        )
        result = Runner.run_sync(agent, context)
        results.append(f"Step {i+1}: {result.final_output}")

    return "\n\n".join(results)

# First run fails at step 4, second run restarts from step 1 (wasteful)
for attempt in range(2):
    print(f"\n=== Attempt {attempt+1} ===")
    try:
        result = run_pipeline("Q3 2024 sales data shows 15% growth in APAC region with declining margins in Europe.")
        print(f"\nFinal Result:\n{result}")
        break
    except RuntimeError as e:
        print(f"Failed: {e}")

## 5. Your Solution

Modify the code above or write your fixed version below.

In [ ]:
# Write your solution here


## 6. Hints

1. In LangGraph, use a SqliteSaver or MemorySaver as the checkpointer when compiling the graph — it automatically persists state after each node
2. Simulate failure by raising an exception in step 4 on the first attempt, then resuming the graph with the same thread_id
3. Store step results in the state dict so that completed steps are not re-executed on resume


## 7. Evaluation Criteria

Check your solution against these criteria:

- Checkpoints after each step
- Resumes from failure point
- Persistent checkpoints
- Correct final result
